# Night-Shift Factory Scheduling

**Artificial Intelligence — Project 01**

This notebook implements an agent capable of generating a weekly night-shift production schedule while respecting machine, operator and production-line constraints.

## 1. Introduction

Factories that run a night shift need to fit a fixed list of weekly tasks into a fixed number of two-hour time slots, while respecting who can do the work, which machines can do it, and when both are actually available. Doing this by hand does not scale well: a small change to one operator's availability can ripple through the whole week, and it is easy to miss a conflict.

This project treats the scheduling problem as a **Constraint Satisfaction Problem (CSP)**:

- the **variables** are the individual task executions that must be scheduled during the week;
- the **domains** are the (slot, machine) combinations each execution could use;
- the **constraints** encode the factory's rules about what a valid schedule looks like.

Some rules can never be broken — a machine cannot run two tasks at once, an operator cannot work while unavailable. These are **hard constraints**, and any schedule that breaks one is rejected outright. Other rules describe *preferences* rather than requirements — for example, spreading a line's work across fewer nights is nicer, but a schedule that fails to do so is still usable. These are **soft constraints**, handled separately with a penalty score once a hard-constraint-valid schedule has been found.

The notebook builds the CSP with the `python-constraint` library, solves it against a small hand-built dataset (`factory_tiny.txt`) as a step-by-step walkthrough, and then reuses the same logic on a larger, more realistic dataset (`factory_real.txt`).


### Authorship

This was a work developed by: 
- Luís Costa
- Student no. **18650**

## 2. Environment and imports

This section imports the Python libraries required to read and process the factory configuration files.

In [92]:
from pathlib import Path

In [93]:
dataset_path = Path("factory_tiny.txt")

print("File exists:", dataset_path.exists())
print("File path:", dataset_path.resolve())

File exists: True
File path: /Users/luiscosta/Desktop/School/IA/Work/IA25_p01/factory_tiny.txt


In [94]:
with dataset_path.open("r", encoding="utf-8") as file:
    factory_text = file.read()

## 3. Dataset parsing

The dataset is divided into sections identified by the markers:

- `#lt`
- `#tue`
- `#to`
- `#mt`
- `#im`
- `#io`
- `#rm`
- `#rt`
- `#machines`

The following function reads the file line by line and groups its contents according to those markers.

In [95]:
def split_into_sections(text):
    sections = {}
    current_section = None

    for raw_line in text.splitlines():
        line = raw_line.strip()

        if not line:
            continue

        # the symbol '#' starts a new section
        if line.startswith("#"):
            current_section = line.split()[0]
            sections[current_section] = []
            continue

        if current_section == "#head":
            continue

        if current_section is not None:
            sections[current_section].append(line)

    return sections

In [96]:
sections = split_into_sections(factory_text)

## 4. Converting sections into structured data

The raw lines are converted into Python dictionaries and lists.

In [97]:
def parse_mapping(lines):

    result = {}

    for line in lines:
        parts = line.split()

        key = parts[0]
        values = parts[1:]

        result[key] = values
    
    return result

In [98]:
lines_to_tasks = parse_mapping(sections["#lt"])
operators_to_tasks = parse_mapping(sections["#to"])
machines_to_tasks = parse_mapping(sections["#mt"])
machine_unavailability = parse_mapping(sections["#im"])
operator_unavailability = parse_mapping(sections["#io"])

In [99]:
def convert_values_to_integers(mapping):
    return {
        key: [int(value) for value in values]
        for key, values in mapping.items()
    }

In [100]:
machine_unavailability = convert_values_to_integers(
    machine_unavailability
)

operator_unavailability = convert_values_to_integers(
    operator_unavailability
)

In [101]:
mandatory_machine = {}

for line in sections["#rm"]:
    task, machine = line.split()
    mandatory_machine[task] = machine

In [102]:
remote_runs = []

for line in sections["#rt"]:
    task, run_number = line.split()

    remote_runs.append(
        (task, int(run_number))
    )

In [103]:
machines = [
    line.strip()
    for line in sections["#machines"]
]

In [104]:
tasks_with_one_run = []

for line in sections["#tue"]:
    tasks_with_one_run.extend(line.split())

## 5. Dataset inspection

Before building the CSP, it's worth checking that the sections were parsed as expected. The cell below prints each structure produced so far: production lines and their tasks, operators, machines, maintenance windows, operator unavailability, mandatory-machine restrictions, remote runs, the list of machines, and any single-run tasks.

In [105]:
print("Production lines:")
print(lines_to_tasks)

print("\nOperators:")
print(operators_to_tasks)

print("\nMachines and supported tasks:")
print(machines_to_tasks)

print("\nMachine unavailability:")
print(machine_unavailability)

print("\nOperator unavailability:")
print(operator_unavailability)

print("\nMandatory machine assignments:")
print(mandatory_machine)

print("\nRemote runs:")
print(remote_runs)

print("\nAvailable machines:")
print(machines)

print("\nTasks with only one run:")
print(tasks_with_one_run)

Production lines:
{'L01': ['T11', 'T12', 'T13', 'T14', 'T15'], 'L02': ['T21', 'T22', 'T23', 'T24', 'T25'], 'L03': ['T31', 'T32', 'T33', 'T34', 'T35']}

Operators:
{'ana': ['T11', 'T21', 'T22', 'T31'], 'rui': ['T12', 'T23', 'T32'], 'ze': ['T13', 'T14', 'T24', 'T33'], 'lia': ['T15', 'T25', 'T34', 'T35']}

Machines and supported tasks:
{'M01': ['T11', 'T12', 'T13', 'T21', 'T22', 'T31', 'T32', 'T33'], 'M02': ['T14', 'T15', 'T23', 'T24', 'T25', 'T34', 'T35'], 'M03': ['T11', 'T13', 'T21', 'T24', 'T31', 'T34']}

Machine unavailability:
{'M02': [13, 14, 15, 16, 17, 18, 19, 20], 'M03': [1, 2, 3, 4]}

Operator unavailability:
{'lia': [9, 10, 11, 12, 17, 18, 19, 20]}

Mandatory machine assignments:
{'T14': 'M02', 'T22': 'M01'}

Remote runs:
[('T21', 2), ('T31', 2)]

Available machines:
['M01', 'M02', 'M03']

Tasks with only one run:
[]


## 6. Agent design and CSP data preparation

The developed system can be considered a goal-based and utility-based software agent.

The agent receives the factory configuration as its input, including production lines, tasks, machines, operators, availability periods, maintenance periods and special restrictions.

Its main goal is to generate a complete weekly schedule that satisfies all mandatory restrictions.

Among the feasible schedules, the agent uses an evaluation function to distinguish schedules of different quality. This utility measure is represented by the soft-constraint penalty score, where lower values indicate more desirable schedules.

The agent does not use machine learning because the problem does not involve prediction from historical examples. It uses automated reasoning and constraint-based search.

The cells below build the reverse-lookup mappings (task → line, task → operator, allowed machines per task) that the CSP setup in the next section depends on.

In [106]:
task_to_line = {}

for line_name, tasks in lines_to_tasks.items():
    for task in tasks:
        task_to_line[task] = line_name

In [107]:
task_to_operator = {}

for operator_name, tasks in operators_to_tasks.items():
    for task in tasks:
        task_to_operator[task] = operator_name

In [108]:
task_to_machines = {}

for machine_name, tasks in machines_to_tasks.items():
    for task in tasks:
        if task not in task_to_machines:
            task_to_machines[task] = []
        task_to_machines[task].append(machine_name)

In [109]:
allowed_machines_by_task = {}

for task, compatible_machines in task_to_machines.items():
    if task in mandatory_machine:
        allowed_machines_by_task[task] = [
            mandatory_machine[task]
        ]
    else:
        allowed_machines_by_task[task] = compatible_machines.copy()

In [110]:
tasks = list(task_to_line.keys())

task_runs = []

for task in tasks:
    # Tasks listed in #tue run once a week; every other task runs twice.
    # Both datasets currently list #tue as empty, so the single-run branch
    # is unexercised, but the logic handles it correctly if it's ever used.
    number_of_runs = 1 if task in tasks_with_one_run else 2

    for run_number in range(1, number_of_runs + 1):
        task_runs.append((task, run_number))

In [111]:
slots = list(range(1,36))

nights = {
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday"
}

def get_night_number(slot):
    return((slot - 1) // 7) + 1

def get_block_in_night(slot):
    return((slot -1) % 7) + 1

## 7. CSP formulation

The night-shift factory scheduling problem was modelled as a Constraint Satisfaction Problem (CSP).

A CSP is defined by a set of variables, the domain of possible values for each variable, and a set of constraints that determine which assignments are valid. A complete and consistent assignment represents a feasible solution to the problem.

### Variables

Each weekly execution of a task is represented as a separate variable.

For example:

- `("T11", 1)` represents the first weekly execution of task `T11`;
- `("T11", 2)` represents the second weekly execution of task `T11`.

This distinction is necessary because most tasks must be executed twice during the week.


### Domains

The value assigned to each variable is a pair with the following structure:

`(slot, machine)`

For example:

`(12, "M01")` means that the task-run is scheduled in slot 12 and uses machine `M01`.

Each task-run initially receives a domain formed by combining the 35 weekly slots with the machines capable of executing that task.

Before adding the variables to the CSP, the domains are filtered to remove assignments that are already known to be invalid, including:

- slots in which the corresponding operator is unavailable;
- slots in which the selected machine is undergoing maintenance;
- machines that do not support the task;
- machines excluded by a mandatory machine restriction.

This early domain filtering reduces the search space and prevents the solver from considering assignments that can never be part of a valid schedule.


### Hard constraints

These hard constraints are mandatory:

1. Each task-run must be assigned to one valid slot and machine.
2. A task can only use a compatible machine.
3. Tasks with a mandatory machine restriction must use the specified machine.
4. A machine cannot execute two tasks during the same slot.
5. A production line cannot execute two tasks during the same slot.
6. An operator cannot execute two tasks during the same slot.
7. A task cannot be scheduled while its operator is unavailable.
8. A machine cannot be used during a maintenance period.
9. A production line cannot execute more than three task-runs during the same night.
10. The first execution of a task must occur before its second execution.
11. The remote task-runs must be scheduled during the same night.

### Soft constraints

After generating feasible solutions, an evaluation function is used to compare their quality.

The following soft constraints are considered:

1. The two weekly executions of the same task should preferably occur on different nights.
2. Each production line should preferably operate during no more than four nights.
3. Tasks belonging to the same production line should preferably be consecutive during each night.
4. Each production line should preferably use as few different machines as possible.

These preferences do not invalidate a schedule. Instead, each violation adds a penalty to the solution score. A lower score represents a better schedule.


### Search and evaluation strategy

The CSP solver is responsible for finding schedules that satisfy every hard constraint.

Because the number of feasible schedules can be very large, a big and long enumeration is not practical. The implementation evaluates solutions until one of the following limits is reached:

- a maximum number of candidate solutions;
- a maximum execution time.

The selected result is the feasible solution with the lowest score among the candidates evaluated. Therefore, it is described as the best solution found within the search limits and not as a guaranteed global optimum.

In [112]:
%pip install python-constraint

Note: you may need to restart the kernel to use updated packages.


In [113]:
from constraint import Problem

problem = Problem()

task_run_domains = {}

for task_run in task_runs:
    task, run_number = task_run

    operator = task_to_operator[task]
    domain = []

    unavailable_operator_slots = operator_unavailability.get(
        operator,
        []
    )

    for slot in slots:
        if slot in unavailable_operator_slots:
            continue
        
        for machine in allowed_machines_by_task[task]:
            unavailable_machine_slots = machine_unavailability.get(
                machine,
                []
            )

            if slot in unavailable_machine_slots:
                continue

            domain.append((slot, machine))
    
    task_run_domains[task_run] = domain


for task_run, domain in task_run_domains.items():
    problem.addVariable(task_run, domain)

In [114]:
print("Number of tasks:", len(tasks))
print("Number of task-runs:", len(task_runs))
print("Number of slots:", len(slots))
print("Number of machines:", len(machines))

print("\nDomain sizes:")

for task_run in task_runs:
    print(
        task_run,
        "→",
        len(task_run_domains[task_run]),
        "possible assignments"
    )

Number of tasks: 15
Number of task-runs: 30
Number of slots: 35
Number of machines: 3

Domain sizes:
('T11', 1) → 66 possible assignments
('T11', 2) → 66 possible assignments
('T12', 1) → 35 possible assignments
('T12', 2) → 35 possible assignments
('T13', 1) → 66 possible assignments
('T13', 2) → 66 possible assignments
('T14', 1) → 27 possible assignments
('T14', 2) → 27 possible assignments
('T15', 1) → 23 possible assignments
('T15', 2) → 23 possible assignments
('T21', 1) → 66 possible assignments
('T21', 2) → 66 possible assignments
('T22', 1) → 35 possible assignments
('T22', 2) → 35 possible assignments
('T23', 1) → 27 possible assignments
('T23', 2) → 27 possible assignments
('T24', 1) → 58 possible assignments
('T24', 2) → 58 possible assignments
('T25', 1) → 23 possible assignments
('T25', 2) → 23 possible assignments
('T31', 1) → 66 possible assignments
('T31', 2) → 66 possible assignments
('T32', 1) → 35 possible assignments
('T32', 2) → 35 possible assignments
('T33', 1) 

## 8. Hard constraints

Any assignment that violates one of these rules is considered invalid.

In [115]:
def first_run_before_second_run(assignment_1, assignment_2):
    slot_1, machine_1 = assignment_1
    slot_2, machine_2 = assignment_2

    # Enforcing slot_1 < slot_2 (rather than just !=) breaks the run_1/run_2
    # symmetry: run 1 is now defined as whichever execution happens first in
    # the week. This also gives "#rt" entries like "T21 2" a stable meaning
    # — the *second* weekly execution — instead of an arbitrary label.
    return slot_1 < slot_2

for task in tasks:
    run_1 = (task, 1)
    run_2 = (task, 2)

    if run_1 in task_runs and run_2 in task_runs:
        problem.addConstraint(
            first_run_before_second_run,
            [run_1, run_2]
        )

In [116]:
def task_run_pairs(task_runs):
    for index_1 in range(len(task_runs)):
        for index_2 in range(index_1 + 1, len(task_runs)):
            yield task_runs[index_1], task_runs[index_2]

def different_time_slots(assignment_1, assignment_2):
    slot_1, machine_1 = assignment_1
    slot_2, machine_2 = assignment_2

    return slot_1 != slot_2

line_constraints_added = 0

for task_run_1, task_run_2 in task_run_pairs(task_runs):
    task_1, run_number_1 = task_run_1
    task_2, run_number_2 = task_run_2

    if task_to_line[task_1] == task_to_line[task_2]:
        problem.addConstraint(
            different_time_slots,
            [task_run_1, task_run_2]
        )

        line_constraints_added += 1

def machines_do_not_overlap(assignment_1, assignment_2):
    slot_1, machine_1 = assignment_1
    slot_2, machine_2 = assignment_2

    if slot_1 != slot_2:
        return True

    return machine_1 != machine_2

machine_constraints_added = 0

for task_run_1, task_run_2 in task_run_pairs(task_runs):
    task_1, _ = task_run_1
    task_2, _ = task_run_2

    machines_1 = set(allowed_machines_by_task[task_1])
    machines_2 = set(allowed_machines_by_task[task_2])

    # No shared machine possible → constraint can never fire, skip it
    if not machines_1 & machines_2:
        continue

    problem.addConstraint(
        machines_do_not_overlap,
        [task_run_1, task_run_2]
    )

    machine_constraints_added += 1

operator_constraints_added = 0

for task_run_1, task_run_2 in task_run_pairs(task_runs):
    task_1, run_number_1 = task_run_1
    task_2, run_number_2 = task_run_2

    operator_1 = task_to_operator[task_1]
    operator_2 = task_to_operator[task_2]

    if operator_1 == operator_2:
        problem.addConstraint(
            different_time_slots,
            [task_run_1, task_run_2]
        )

        operator_constraints_added += 1

In [117]:
from constraint import Constraint

class MaxTasksPerNightConstraint(Constraint):
    def __init__(self, night, max_count=3):
        self._night = night
        self._max_count = max_count

    def __call__(self, variables, domains, assignments, forwardcheck=False):
        night = self._night
        max_count = self._max_count

        count = 0
        for variable in variables:
            if variable in assignments:
                slot, machine = assignments[variable]
                if get_night_number(slot) == night:
                    count += 1

        if count > max_count:
            return False

        if forwardcheck and count == max_count:
            # This night is already full: remove it from the remaining domains
            for variable in variables:
                if variable not in assignments:
                    domain = domains[variable]
                    for value in domain[:]:
                        slot, machine = value
                        if get_night_number(slot) == night:
                            domain.hideValue(value)
                    if not domain:
                        return False

        return True

In [118]:
line_to_task_runs = {}

for line_name in lines_to_tasks:
    line_to_task_runs[line_name] = []

for task_run in task_runs:
    task, run_number = task_run
    line_name = task_to_line[task]

    line_to_task_runs[line_name].append(task_run)


for line_name, runs_on_line in line_to_task_runs.items():
    for night in nights:
        problem.addConstraint(
            MaxTasksPerNightConstraint(night, max_count=3),
            runs_on_line
        )

### Remote-task constraint (challenge)

Per the dataset's `#rt` section, a small number of task-runs (limited to three) are teleoperated/remote each week. The challenge hard constraint requires that whenever remote runs are scheduled, they must all fall on the same night — regardless of which line or machine they belong to.

This relies on the run-order fix above: `#rt` identifies a remote run by task and run number (e.g. `T21 2`), and that label is only meaningful once run 2 is guaranteed to be the run that actually happens second in the week.

In [119]:
remote_task_runs = [
    (task, run_number)
    for task, run_number in remote_runs
]

print("Remote task-runs:", remote_task_runs)

if len(remote_task_runs) > 3:
    print(
        "Warning: more than three remote task-runs found; "
        "the guide assumes a maximum of three."
    )


def remote_tasks_same_night(*assignments):
    nights_used = {
        get_night_number(slot)
        for slot, machine in assignments
    }

    return len(nights_used) <= 1


if len(remote_task_runs) > 1:
    problem.addConstraint(remote_tasks_same_night, remote_task_runs)

Remote task-runs: [('T21', 2), ('T31', 2)]


## 9. Executing the CSP solver (factory_tiny.txt)

The solver is executed to obtain a valid weekly schedule.

At this stage, only the hard constraints are considered. The first valid solution found is returned and its execution time is measured.

In [120]:
from time import perf_counter

In [121]:
start_time = perf_counter()

first_solution = problem.getSolution()

end_time = perf_counter()

execution_time = end_time - start_time

In [122]:
print("Solution found:", first_solution is not None)
print(f"Execution time: {execution_time:.4f} seconds")
first_solution

Solution found: True
Execution time: 0.0045 seconds


{('T34', 1): (34, 'M03'),
 ('T34', 2): (35, 'M03'),
 ('T24', 1): (34, 'M02'),
 ('T24', 2): (35, 'M02'),
 ('T21', 2): (33, 'M03'),
 ('T31', 2): (32, 'M03'),
 ('T21', 1): (28, 'M03'),
 ('T31', 1): (27, 'M03'),
 ('T11', 1): (34, 'M01'),
 ('T11', 2): (35, 'M01'),
 ('T13', 1): (28, 'M01'),
 ('T13', 2): (33, 'M01'),
 ('T22', 1): (25, 'M01'),
 ('T22', 2): (26, 'M01'),
 ('T33', 1): (23, 'M01'),
 ('T33', 2): (24, 'M01'),
 ('T25', 1): (8, 'M02'),
 ('T25', 2): (21, 'M02'),
 ('T35', 1): (6, 'M02'),
 ('T35', 2): (7, 'M02'),
 ('T15', 1): (26, 'M02'),
 ('T15', 2): (27, 'M02'),
 ('T14', 1): (11, 'M02'),
 ('T14', 2): (12, 'M02'),
 ('T12', 1): (20, 'M01'),
 ('T12', 2): (21, 'M01'),
 ('T32', 1): (18, 'M01'),
 ('T32', 2): (19, 'M01'),
 ('T23', 1): (9, 'M02'),
 ('T23', 2): (10, 'M02')}

In [123]:
sorted_solution = sorted(
    first_solution.items(),
    key=lambda item: item[1][0]
)

for task_run, assignment in sorted_solution:
    task, run_number = task_run
    slot, machine = assignment

    print(
        f"Slot {slot:2} | "
        f"Task {task} | "
        f"Run {run_number} | "
        f"Line {task_to_line[task]} | "
        f"Operator {task_to_operator[task]} | "
        f"Machine {machine}"
    )

block_times = {
    1: "18:00–20:00",
    2: "20:00–22:00",
    3: "22:00–00:00",
    4: "00:00–02:00",
    5: "02:00–04:00",
    6: "05:00–07:00",
    7: "07:00–09:00"
}

def get_slot_description(slot):
    night_number = get_night_number(slot)
    block_number = get_block_in_night(slot)

    return (
        nights[night_number],
        block_times[block_number]
    )


for task_run, assignment in sorted_solution:
    task, run_number = task_run
    slot, machine = assignment

    night_name, time_period = get_slot_description(slot)

    print(
        f"{night_name:9} | "
        f"{time_period} | "
        f"{task} (run {run_number}) | "
        f"Line {task_to_line[task]} | "
        f"{machine} | "
        f"Operator {task_to_operator[task]}"
    )

Slot  6 | Task T35 | Run 1 | Line L03 | Operator lia | Machine M02
Slot  7 | Task T35 | Run 2 | Line L03 | Operator lia | Machine M02
Slot  8 | Task T25 | Run 1 | Line L02 | Operator lia | Machine M02
Slot  9 | Task T23 | Run 1 | Line L02 | Operator rui | Machine M02
Slot 10 | Task T23 | Run 2 | Line L02 | Operator rui | Machine M02
Slot 11 | Task T14 | Run 1 | Line L01 | Operator ze | Machine M02
Slot 12 | Task T14 | Run 2 | Line L01 | Operator ze | Machine M02
Slot 18 | Task T32 | Run 1 | Line L03 | Operator rui | Machine M01
Slot 19 | Task T32 | Run 2 | Line L03 | Operator rui | Machine M01
Slot 20 | Task T12 | Run 1 | Line L01 | Operator rui | Machine M01
Slot 21 | Task T25 | Run 2 | Line L02 | Operator lia | Machine M02
Slot 21 | Task T12 | Run 2 | Line L01 | Operator rui | Machine M01
Slot 23 | Task T33 | Run 1 | Line L03 | Operator ze | Machine M01
Slot 24 | Task T33 | Run 2 | Line L03 | Operator ze | Machine M01
Slot 25 | Task T22 | Run 1 | Line L02 | Operator ana | Machine M01

### Solution validation

The following function independently checks whether the generated solution respects the principal hard constraints.

In [124]:
def validate_solution(solution):
    errors = []

    scheduled_runs = list(solution.items())

    for index_1 in range(len(scheduled_runs)):
        task_run_1, assignment_1 = scheduled_runs[index_1]
        task_1, run_number_1 = task_run_1
        slot_1, machine_1 = assignment_1

        operator_1 = task_to_operator[task_1]
        line_1 = task_to_line[task_1]

        # Check allowed machine
        if machine_1 not in allowed_machines_by_task[task_1]:
            errors.append(
                f"{task_run_1} uses invalid machine {machine_1}"
            )

        # Check operator availability
        unavailable_operator_slots = operator_unavailability.get(
            operator_1,
            []
        )

        if slot_1 in unavailable_operator_slots:
            errors.append(
                f"{task_run_1} is scheduled while "
                f"operator {operator_1} is unavailable"
            )

        # Check machine maintenance
        unavailable_machine_slots = machine_unavailability.get(
            machine_1,
            []
        )

        if slot_1 in unavailable_machine_slots:
            errors.append(
                f"{task_run_1} uses machine {machine_1} "
                f"during maintenance"
            )

        for index_2 in range(index_1 + 1, len(scheduled_runs)):
            task_run_2, assignment_2 = scheduled_runs[index_2]
            task_2, run_number_2 = task_run_2
            slot_2, machine_2 = assignment_2

            operator_2 = task_to_operator[task_2]
            line_2 = task_to_line[task_2]

            if slot_1 != slot_2:
                continue

            if line_1 == line_2:
                errors.append(
                    f"Line conflict in slot {slot_1}: "
                    f"{task_run_1} and {task_run_2}"
                )

            if machine_1 == machine_2:
                errors.append(
                    f"Machine conflict in slot {slot_1}: "
                    f"{task_run_1} and {task_run_2} "
                    f"use {machine_1}"
                )

            if operator_1 == operator_2:
                errors.append(
                    f"Operator conflict in slot {slot_1}: "
                    f"{task_run_1} and {task_run_2} "
                    f"use operator {operator_1}"
                )

    # Check maximum of three tasks per line per night
    for line_name, runs_on_line in line_to_task_runs.items():
        counts_by_night = {
            1: 0,
            2: 0,
            3: 0,
            4: 0,
            5: 0
        }

        for task_run in runs_on_line:
            slot, machine = solution[task_run]
            night_number = get_night_number(slot)

            counts_by_night[night_number] += 1

        for night_number, count in counts_by_night.items():
            if count > 3:
                errors.append(
                    f"Line {line_name} has {count} tasks "
                    f"on {nights[night_number]}"
                )

    return errors


validation_errors = validate_solution(first_solution)

print("Number of validation errors:", len(validation_errors))

for error in validation_errors:
    print("-", error)

Number of validation errors: 0


## 10. Counting solutions

`problem.getSolution()` only returns the first valid assignment the solver happens to find. To report "how many solutions exist" we need to enumerate with `problem.getSolutionIter()` instead.

However, the hard constraints alone allow an enormous number of valid-but-not-great schedules (soft constraints are not enforced here), so a full enumeration is not practical even for the tiny dataset. Following the guide's own distinction between an **optimal solution** and the **best solution found in a given amount of time**, we cap the search by a time limit and a solution-count limit, and treat the result as a lower bound rather than an exhaustive count.

In [125]:
SOLUTION_SEARCH_TIME_LIMIT = 180    # seconds
SOLUTION_SEARCH_COUNT_LIMIT = 5000000  # maximum solutions kept in memory

found_solutions = []
search_was_exhaustive = True

solution_search_start = perf_counter()

for solution in problem.getSolutionIter():
    found_solutions.append(solution)

    if len(found_solutions) >= SOLUTION_SEARCH_COUNT_LIMIT:
        search_was_exhaustive = False
        break

    if perf_counter() - solution_search_start >= SOLUTION_SEARCH_TIME_LIMIT:
        search_was_exhaustive = False
        break

solution_search_time = perf_counter() - solution_search_start

In [126]:
print("Solutions found:", len(found_solutions))
print("Search exhaustive:", search_was_exhaustive)
print(f"Search time: {solution_search_time:.4f} seconds")

if not search_was_exhaustive:
    print(
        "\nNote: the search space is far larger than this cap. This count is a "
        "lower bound on the number of valid (hard-constraint-satisfying) schedules, "
        "not an exhaustive count."
    )

Solutions found: 5000000
Search exhaustive: False
Search time: 121.7756 seconds

Note: the search space is far larger than this cap. This count is a lower bound on the number of valid (hard-constraint-satisfying) schedules, not an exhaustive count.


## 11. Soft constraints and scoring

A solution may violate the following preferences and still be valid; each violation adds a penalty. The best schedule is the one with the lowest total penalty among the solutions found above.

1. **Different nights per task** — the two runs of the same task should not fall on the same night (penalty 20 per violation).
2. **At most four nights per line** — a line should ideally have one free night during the week (penalty 10 if a line uses all five nights).
3. **Consecutive tasks per night** — a line's tasks on a given night should occupy consecutive slots rather than leaving gaps (penalty 5 per gap slot).
4. **Fewer machines per line** (challenge) — a line should use as few distinct machines as possible across the week (penalty 2 per machine beyond the first).

In [127]:
def score_same_night_repeats(solution):
    penalty = 0

    for task in tasks:
        run_1 = (task, 1)
        run_2 = (task, 2)

        if run_1 in solution and run_2 in solution:
            night_1 = get_night_number(solution[run_1][0])
            night_2 = get_night_number(solution[run_2][0])

            if night_1 == night_2:
                penalty += 20

    return penalty


def score_nights_used_per_line(solution):
    penalty = 0

    for line_name, runs_on_line in line_to_task_runs.items():
        nights_used = {
            get_night_number(solution[task_run][0])
            for task_run in runs_on_line
        }

        if len(nights_used) >= 5:
            penalty += 10

    return penalty


def score_non_consecutive_tasks(solution):
    penalty = 0

    for line_name, runs_on_line in line_to_task_runs.items():
        blocks_by_night = {}

        for task_run in runs_on_line:
            slot, machine = solution[task_run]
            night = get_night_number(slot)
            block = get_block_in_night(slot)

            blocks_by_night.setdefault(night, []).append(block)

        for night, blocks in blocks_by_night.items():
            blocks = sorted(blocks)
            span = blocks[-1] - blocks[0] + 1
            gaps = span - len(blocks)

            penalty += 5 * gaps

    return penalty


def score_machines_used_per_line(solution):
    penalty = 0

    for line_name, runs_on_line in line_to_task_runs.items():
        machines_used = {
            solution[task_run][1]
            for task_run in runs_on_line
        }

        penalty += 2 * (len(machines_used) - 1)

    return penalty


def evaluate_solution(solution):
    return (
        score_same_night_repeats(solution)
        + score_nights_used_per_line(solution)
        + score_non_consecutive_tasks(solution)
        + score_machines_used_per_line(solution)
    )

In [128]:
scoring_start = perf_counter()

best_solution = min(found_solutions, key=evaluate_solution)
best_score = evaluate_solution(best_solution)

scoring_time = perf_counter() - scoring_start

print("Candidate solutions evaluated:", len(found_solutions))
print("Best score found:", best_score)
print(f"Scoring time: {scoring_time:.4f} seconds")

print("\nBest solution penalty breakdown:")
print("- Same-night repeats:  ", score_same_night_repeats(best_solution))
print("- Lines using 5 nights:", score_nights_used_per_line(best_solution))
print("- Non-consecutive gaps:", score_non_consecutive_tasks(best_solution))
print("- Extra machines used: ", score_machines_used_per_line(best_solution))

Candidate solutions evaluated: 5000000
Best score found: 180
Scoring time: 59.6006 seconds

Best solution penalty breakdown:
- Same-night repeats:   140
- Lines using 5 nights: 10
- Non-consecutive gaps: 20
- Extra machines used:  10


In [129]:
best_sorted_solution = sorted(
    best_solution.items(),
    key=lambda item: item[1][0]
)

for task_run, assignment in best_sorted_solution:
    task, run_number = task_run
    slot, machine = assignment

    night_name, time_period = get_slot_description(slot)

    print(
        f"{night_name:9} | "
        f"{time_period} | "
        f"{task} (run {run_number}) | "
        f"Line {task_to_line[task]} | "
        f"{machine} | "
        f"Operator {task_to_operator[task]}"
    )

best_solution_errors = validate_solution(best_solution)
print("\nNumber of validation errors:", len(best_solution_errors))

Monday    | 20:00–22:00 | T23 (run 1) | Line L02 | M02 | Operator rui
Monday    | 02:00–04:00 | T14 (run 1) | Line L01 | M02 | Operator ze
Monday    | 02:00–04:00 | T32 (run 1) | Line L03 | M01 | Operator rui
Monday    | 05:00–07:00 | T35 (run 1) | Line L03 | M02 | Operator lia
Monday    | 05:00–07:00 | T12 (run 1) | Line L01 | M01 | Operator rui
Monday    | 07:00–09:00 | T35 (run 2) | Line L03 | M02 | Operator lia
Tuesday   | 18:00–20:00 | T25 (run 1) | Line L02 | M02 | Operator lia
Tuesday   | 18:00–20:00 | T32 (run 2) | Line L03 | M01 | Operator rui
Tuesday   | 20:00–22:00 | T23 (run 2) | Line L02 | M02 | Operator rui
Tuesday   | 22:00–00:00 | T14 (run 2) | Line L01 | M02 | Operator ze
Tuesday   | 00:00–02:00 | T12 (run 2) | Line L01 | M01 | Operator rui
Wednesday | 07:00–09:00 | T25 (run 2) | Line L02 | M02 | Operator lia
Thursday  | 20:00–22:00 | T33 (run 1) | Line L03 | M01 | Operator ze
Thursday  | 22:00–00:00 | T33 (run 2) | Line L03 | M01 | Operator ze
Thursday  | 00:00–02:00 

## 12. Schedule visualization (factory_tiny.txt)

A flat list of `(task_run → slot, machine)` pairs is hard to read. This section renders one weekly grid per production line — nights as columns, the seven 2-hour blocks as rows — similar to the example schedule in the project guide (page 7).

Each cell is color-coded:
- **normal** — a regular task-run.
- **machine-pinned** (`#rm`) — the task is forced onto a specific machine.
- **remote** (`#rt`) — a teleoperated run (also same-night with any other remote runs, per section 8's hard constraint).

In [130]:
from IPython.display import HTML, display

TABLE_STYLE = """
<style>
  .schedule-table { border-collapse: collapse; text-align: center; font-family: sans-serif; margin-bottom: 1.5em; }
  .schedule-table th { padding: 6px 10px; }
  .schedule-table td { padding: 6px 10px; border: 1px solid #d0d5dc; min-width: 70px; }
  .schedule-table .time-label { color: #888; border: none; font-size: 0.85em; }
  .schedule-table .empty { color: #c7ccd3; }
  .schedule-table .normal { background: #eef1f5; }
  .schedule-table .pinned { background: #f6d896; }
  .schedule-table .remote { background: #1f2a44; color: white; }
</style>
"""


def render_schedule_table(solution, line_name):
    runs_on_line = [
        task_run for task_run in solution
        if task_to_line[task_run[0]] == line_name
    ]

    cell_by_position = {}
    for task_run in runs_on_line:
        task, run_number = task_run
        slot, machine = solution[task_run]
        night = get_night_number(slot)
        block = get_block_in_night(slot)
        cell_by_position[(night, block)] = (task, run_number, machine)

    header_cells = "".join(f"<th>{nights[night]}</th>" for night in range(1, 6))

    rows_html = ""
    for block in range(1, 8):
        row_cells = f"<td class='time-label'>{block_times[block]}</td>"

        for night in range(1, 6):
            entry = cell_by_position.get((night, block))

            if entry is None:
                row_cells += "<td class='empty'>—</td>"
                continue

            task, run_number, machine = entry
            css_class = "normal"
            suffix = ""

            if task in mandatory_machine:
                css_class = "pinned"

            if (task, run_number) in remote_task_runs:
                css_class = "remote"
                suffix = "<br><small>· rem</small>"

            row_cells += (
                f"<td class='{css_class}'><b>{task}</b> (run {run_number})"
                f"<br><small>{machine}</small>{suffix}</td>"
            )

        rows_html += f"<tr>{row_cells}</tr>"

    display(HTML(
        TABLE_STYLE
        + f"<h4>Line {line_name}</h4>"
        + f"<table class='schedule-table'><tr><th></th>{header_cells}</tr>{rows_html}</table>"
    ))

### Schedule export helper

The displayed schedules are also exported to SVG image files. Existing files with the same name are deleted before the new files are written, so rerunning the notebook always refreshes the exported schedules.

In [131]:
from xml.sax.saxutils import escape

SCHEDULE_EXPORT_DIR = Path("schedule_exports")


def build_schedule_svg(solution, title):
    line_names = list(lines_to_tasks.keys())

    margin = 24
    title_height = 48
    line_title_height = 28
    row_height = 58
    time_column_width = 96
    day_column_width = 148
    table_gap = 36

    table_width = time_column_width + 5 * day_column_width
    table_height = line_title_height + 8 * row_height
    width = table_width + 2 * margin
    height = title_height + len(line_names) * (table_height + table_gap) + margin

    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        '<style>text { font-family: Arial, sans-serif; fill: #172033; } .small { font-size: 11px; fill: #4b5563; } .title { font-size: 22px; font-weight: 700; } .line-title { font-size: 16px; font-weight: 700; } .header { font-size: 12px; font-weight: 700; } .cell-main { font-size: 12px; font-weight: 700; } .cell-sub { font-size: 10px; fill: #4b5563; }</style>',
        f'<text x="{margin}" y="32" class="title">{escape(title)}</text>'
    ]

    def add_text(x, y, text, css_class, anchor="middle"):
        parts.append(
            f'<text x="{x}" y="{y}" text-anchor="{anchor}" class="{css_class}">{escape(str(text))}</text>'
        )

    y = title_height
    for line_name in line_names:
        parts.append(f'<text x="{margin}" y="{y + 18}" class="line-title">Line {escape(str(line_name))}</text>')

        table_y = y + line_title_height
        parts.append(
            f'<rect x="{margin}" y="{table_y}" width="{table_width}" height="{8 * row_height}" fill="#ffffff" stroke="#cbd5e1"/>'
        )

        for column in range(6):
            column_width = time_column_width if column == 0 else day_column_width
            column_x = margin if column == 0 else margin + time_column_width + (column - 1) * day_column_width
            parts.append(
                f'<rect x="{column_x}" y="{table_y}" width="{column_width}" height="{row_height}" fill="#eef2f7" stroke="#cbd5e1"/>'
            )

        for night in range(1, 6):
            text_x = margin + time_column_width + (night - 1) * day_column_width + day_column_width / 2
            add_text(text_x, table_y + 35, nights[night], "header")

        cell_by_position = {}
        for task_run in solution:
            task, run_number = task_run
            if task_to_line[task] != line_name:
                continue

            slot, machine = solution[task_run]
            night = get_night_number(slot)
            block = get_block_in_night(slot)
            cell_by_position[(night, block)] = (task, run_number, machine)

        for block in range(1, 8):
            row_y = table_y + block * row_height
            parts.append(
                f'<rect x="{margin}" y="{row_y}" width="{time_column_width}" height="{row_height}" fill="#ffffff" stroke="#cbd5e1"/>'
            )
            add_text(margin + time_column_width / 2, row_y + 34, block_times[block], "small")

            for night in range(1, 6):
                cell_x = margin + time_column_width + (night - 1) * day_column_width
                entry = cell_by_position.get((night, block))

                fill = "#f8fafc"
                text_fill = "#172033"
                if entry is not None:
                    task, run_number, machine = entry
                    fill = "#eef1f5"
                    if task in mandatory_machine:
                        fill = "#f6d896"
                    if (task, run_number) in remote_task_runs:
                        fill = "#1f2a44"
                        text_fill = "#ffffff"

                parts.append(
                    f'<rect x="{cell_x}" y="{row_y}" width="{day_column_width}" height="{row_height}" fill="{fill}" stroke="#cbd5e1"/>'
                )

                if entry is None:
                    add_text(cell_x + day_column_width / 2, row_y + 34, "-", "small")
                else:
                    task, run_number, machine = entry
                    remote_label = " remote" if (task, run_number) in remote_task_runs else ""
                    parts.append(
                        f'<text x="{cell_x + day_column_width / 2}" y="{row_y + 23}" text-anchor="middle" class="cell-main" style="fill: {text_fill};">{escape(task)} (run {run_number})</text>'
                    )
                    parts.append(
                        f'<text x="{cell_x + day_column_width / 2}" y="{row_y + 41}" text-anchor="middle" class="cell-sub" style="fill: {text_fill};">{escape(machine)}{remote_label}</text>'
                    )

        y += table_height + table_gap

    parts.append('</svg>')
    return "\n".join(parts)


def export_schedule_svg(solution, output_filename, title):
    SCHEDULE_EXPORT_DIR.mkdir(exist_ok=True)
    output_path = SCHEDULE_EXPORT_DIR / output_filename

    if output_path.exists():
        output_path.unlink()

    output_path.write_text(
        build_schedule_svg(solution, title),
        encoding="utf-8"
    )

    return output_path

In [132]:
print("Best schedule — factory_tiny.txt\n")

for line_name in lines_to_tasks:
    render_schedule_table(best_solution, line_name)

Best schedule — factory_tiny.txt



,Monday,Tuesday,Wednesday,Thursday,Friday
18:00–20:00,—,—,—,—,—
20:00–22:00,—,—,—,—,—
22:00–00:00,—,T14 (run 2)M02,—,—,—
00:00–02:00,—,T12 (run 2)M01,—,—,—
02:00–04:00,T14 (run 1)M02,—,—,T15 (run 1)M02,T13 (run 2)M01
05:00–07:00,T12 (run 1)M01,—,—,T15 (run 2)M02,T11 (run 1)M01
07:00–09:00,—,—,—,T13 (run 1)M01,T11 (run 2)M01


,Monday,Tuesday,Wednesday,Thursday,Friday
18:00–20:00,—,T25 (run 1)M02,—,—,—
20:00–22:00,T23 (run 1)M02,T23 (run 2)M02,—,—,—
22:00–00:00,—,—,—,—,—
00:00–02:00,—,—,—,T22 (run 1)M01,—
02:00–04:00,—,—,—,T22 (run 2)M01,T21 (run 2)M03· rem
05:00–07:00,—,—,—,—,T24 (run 1)M02
07:00–09:00,—,—,T25 (run 2)M02,T21 (run 1)M03,T24 (run 2)M02


,Monday,Tuesday,Wednesday,Thursday,Friday
18:00–20:00,—,T32 (run 2)M01,—,—,—
20:00–22:00,—,—,—,T33 (run 1)M01,—
22:00–00:00,—,—,—,T33 (run 2)M01,—
00:00–02:00,—,—,—,—,T31 (run 2)M03· rem
02:00–04:00,T32 (run 1)M01,—,—,—,—
05:00–07:00,T35 (run 1)M02,—,—,T31 (run 1)M03,T34 (run 1)M03
07:00–09:00,T35 (run 2)M02,—,—,—,T34 (run 2)M03


In [133]:
tiny_svg_path = export_schedule_svg(
    best_solution,
    "factory_tiny_schedule.svg",
    "Best schedule - factory_tiny.txt"
)

print(f"Schedule exported to: {tiny_svg_path}")

Schedule exported to: schedule_exports/factory_tiny_schedule.svg


## 13. Execution with factory_real.txt

Everything above (sections 6–11) was written as a step-by-step walkthrough against `factory_tiny.txt`, one concept per cell, so the modeling decisions are easy to follow.

To run the same pipeline against `factory_real.txt` without duplicating all of those cells, `run_scheduler` below wraps the same steps — parsing, reverse mappings, CSP construction, hard constraints, solving, capped enumeration and scoring — into a single reusable function. It reuses the exact same constraint functions and scoring functions defined above (`first_run_before_second_run`, `different_time_slots`, `machines_do_not_overlap`, `MaxTasksPerNightConstraint`, `remote_tasks_same_night`, `validate_solution`, `evaluate_solution`, ...), and updates the same module-level variables they depend on (`task_to_line`, `allowed_machines_by_task`, `line_to_task_runs`, etc.), so calling it for the real dataset produces directly comparable results.

In [134]:
def run_scheduler(dataset_file, solution_count_limit=SOLUTION_SEARCH_COUNT_LIMIT, search_time_limit=SOLUTION_SEARCH_TIME_LIMIT):
    global sections, lines_to_tasks, operators_to_tasks, machines_to_tasks
    global machine_unavailability, operator_unavailability, mandatory_machine
    global remote_runs, machines, tasks_with_one_run
    global task_to_line, task_to_operator, task_to_machines, allowed_machines_by_task
    global tasks, task_runs, problem, task_run_domains, line_to_task_runs, remote_task_runs

    path = Path(dataset_file)

    with path.open("r", encoding="utf-8") as file:
        factory_text = file.read()

    sections = split_into_sections(factory_text)

    lines_to_tasks = parse_mapping(sections["#lt"])
    operators_to_tasks = parse_mapping(sections["#to"])
    machines_to_tasks = parse_mapping(sections["#mt"])
    machine_unavailability = convert_values_to_integers(parse_mapping(sections["#im"]))
    operator_unavailability = convert_values_to_integers(parse_mapping(sections["#io"]))

    mandatory_machine = {}
    for line in sections["#rm"]:
        task, machine = line.split()
        mandatory_machine[task] = machine

    remote_runs = []
    for line in sections["#rt"]:
        task, run_number = line.split()
        remote_runs.append((task, int(run_number)))

    machines = [line.strip() for line in sections["#machines"]]

    tasks_with_one_run = []
    for line in sections["#tue"]:
        tasks_with_one_run.extend(line.split())

    # Reverse mappings
    task_to_line = {}
    for line_name, line_tasks in lines_to_tasks.items():
        for task in line_tasks:
            task_to_line[task] = line_name

    task_to_operator = {}
    for operator_name, operator_tasks in operators_to_tasks.items():
        for task in operator_tasks:
            task_to_operator[task] = operator_name

    task_to_machines = {}
    for machine_name, machine_tasks in machines_to_tasks.items():
        for task in machine_tasks:
            task_to_machines.setdefault(task, []).append(machine_name)

    allowed_machines_by_task = {}
    for task, compatible_machines in task_to_machines.items():
        if task in mandatory_machine:
            allowed_machines_by_task[task] = [mandatory_machine[task]]
        else:
            allowed_machines_by_task[task] = compatible_machines.copy()

    tasks = list(task_to_line.keys())

    task_runs = []
    for task in tasks:
        number_of_runs = 1 if task in tasks_with_one_run else 2
        for run_number in range(1, number_of_runs + 1):
            task_runs.append((task, run_number))

    # CSP variables and domains
    problem = Problem()
    task_run_domains = {}

    for task_run in task_runs:
        task, run_number = task_run
        operator = task_to_operator[task]
        domain = []
        unavailable_operator_slots = operator_unavailability.get(operator, [])

        for slot in slots:
            if slot in unavailable_operator_slots:
                continue
            for machine in allowed_machines_by_task[task]:
                unavailable_machine_slots = machine_unavailability.get(machine, [])
                if slot in unavailable_machine_slots:
                    continue
                domain.append((slot, machine))

        task_run_domains[task_run] = domain

    for task_run, domain in task_run_domains.items():
        problem.addVariable(task_run, domain)

    # Hard constraints (same rules as sections 6, reusing the same functions)
    for task in tasks:
        run_1 = (task, 1)
        run_2 = (task, 2)
        if run_1 in task_runs and run_2 in task_runs:
            problem.addConstraint(first_run_before_second_run, [run_1, run_2])

    for task_run_1, task_run_2 in task_run_pairs(task_runs):
        task_1, _ = task_run_1
        task_2, _ = task_run_2
        if task_to_line[task_1] == task_to_line[task_2]:
            problem.addConstraint(different_time_slots, [task_run_1, task_run_2])

    for task_run_1, task_run_2 in task_run_pairs(task_runs):
        task_1, _ = task_run_1
        task_2, _ = task_run_2
        machines_1 = set(allowed_machines_by_task[task_1])
        machines_2 = set(allowed_machines_by_task[task_2])
        if not machines_1 & machines_2:
            continue
        problem.addConstraint(machines_do_not_overlap, [task_run_1, task_run_2])

    for task_run_1, task_run_2 in task_run_pairs(task_runs):
        task_1, _ = task_run_1
        task_2, _ = task_run_2
        if task_to_operator[task_1] == task_to_operator[task_2]:
            problem.addConstraint(different_time_slots, [task_run_1, task_run_2])

    line_to_task_runs = {line_name: [] for line_name in lines_to_tasks}
    for task_run in task_runs:
        task, _ = task_run
        line_to_task_runs[task_to_line[task]].append(task_run)

    for line_name, runs_on_line in line_to_task_runs.items():
        for night in nights:
            problem.addConstraint(
                MaxTasksPerNightConstraint(night, max_count=3),
                runs_on_line
            )

    remote_task_runs = [(task, run_number) for task, run_number in remote_runs]
    if len(remote_task_runs) > 1:
        problem.addConstraint(remote_tasks_same_night, remote_task_runs)

    # Solve for a single solution
    solve_start = perf_counter()
    solution = problem.getSolution()
    solve_time = perf_counter() - solve_start

    validation_errors = validate_solution(solution) if solution is not None else None

    # Capped enumeration + soft-constraint scoring
    found_solutions_local = []
    exhaustive = True
    search_start = perf_counter()

    if solution is not None:
        for candidate in problem.getSolutionIter():
            found_solutions_local.append(candidate)

            if len(found_solutions_local) >= solution_count_limit:
                exhaustive = False
                break

            if perf_counter() - search_start >= search_time_limit:
                exhaustive = False
                break

    search_time = perf_counter() - search_start

    best = None
    best_score = None
    scoring_time = 0.0

    if found_solutions_local:
        scoring_start = perf_counter()
        best = min(found_solutions_local, key=evaluate_solution)
        best_score = evaluate_solution(best)
        scoring_time = perf_counter() - scoring_start

    return {
        "dataset_path": str(path),
        "num_lines": len(lines_to_tasks),
        "num_tasks": len(tasks),
        "num_task_runs": len(task_runs),
        "num_machines": len(machines),
        "first_solution": solution,
        "solve_time": solve_time,
        "validation_errors": validation_errors,
        "solutions_found": len(found_solutions_local),
        "search_exhaustive": exhaustive,
        "search_time": search_time,
        "best_solution": best,
        "best_score": best_score,
        "scoring_time": scoring_time,
    }

In [135]:
real_results = run_scheduler("factory_real.txt")

print("Dataset:", real_results["dataset_path"])
print("Lines:", real_results["num_lines"])
print("Tasks:", real_results["num_tasks"])
print("Task-runs:", real_results["num_task_runs"])
print("Machines:", real_results["num_machines"])

print("\nSolution found:", real_results["first_solution"] is not None)
print(f"Solve time: {real_results['solve_time']:.4f} seconds")
print("Validation errors:", len(real_results["validation_errors"]))

print("\nSolutions found (capped):", real_results["solutions_found"])
print("Search exhaustive:", real_results["search_exhaustive"])
print(f"Search time: {real_results['search_time']:.4f} seconds")

print("\nBest score found:", real_results["best_score"])
print(f"Scoring time: {real_results['scoring_time']:.4f} seconds")

Dataset: factory_real.txt
Lines: 4
Tasks: 18
Task-runs: 36
Machines: 6

Solution found: True
Solve time: 0.0048 seconds
Validation errors: 0

Solutions found (capped): 5000000
Search exhaustive: False
Search time: 98.5893 seconds

Best score found: 216
Scoring time: 69.0704 seconds


In [136]:
real_best_solution = real_results["best_solution"]

real_best_sorted = sorted(
    real_best_solution.items(),
    key=lambda item: item[1][0]
)

for task_run, assignment in real_best_sorted:
    task, run_number = task_run
    slot, machine = assignment

    night_name, time_period = get_slot_description(slot)

    print(
        f"{night_name:9} | "
        f"{time_period} | "
        f"{task} (run {run_number}) | "
        f"Line {task_to_line[task]} | "
        f"{machine} | "
        f"Operator {task_to_operator[task]}"
    )

Tuesday   | 07:00–09:00 | SOLD (run 1) | Line MON | BAN01 | Operator Helena
Tuesday   | 07:00–09:00 | CTQ (run 1) | Line MAQ | CTQ01 | Operator Bruno
Wednesday | 02:00–04:00 | SOLD (run 2) | Line MON | BAN01 | Operator Helena
Wednesday | 02:00–04:00 | CTQ (run 2) | Line MAQ | CTQ01 | Operator Bruno
Wednesday | 05:00–07:00 | APAR (run 1) | Line MON | BAN01 | Operator Helena
Wednesday | 05:00–07:00 | TORN (run 1) | Line MAQ | CNC01 | Operator Bruno
Wednesday | 05:00–07:00 | REBA (run 1) | Line INJ | INJ01 | Operator Carlos
Wednesday | 05:00–07:00 | PINT (run 1) | Line ACB | CAB01 | Operator Rui
Wednesday | 07:00–09:00 | APAR (run 2) | Line MON | BAN01 | Operator Helena
Wednesday | 07:00–09:00 | TORN (run 2) | Line MAQ | CNC01 | Operator Bruno
Wednesday | 07:00–09:00 | REBA (run 2) | Line INJ | INJ02 | Operator Carlos
Wednesday | 07:00–09:00 | LAVA (run 1) | Line ACB | CAB01 | Operator Rui
Thursday  | 02:00–04:00 | TEST (run 1) | Line MON | BAN01 | Operator Sofia
Thursday  | 02:00–04:00 |

In [137]:
print("Best schedule — factory_real.txt\n")

for line_name in lines_to_tasks:
    render_schedule_table(real_best_solution, line_name)

Best schedule — factory_real.txt



,Monday,Tuesday,Wednesday,Thursday,Friday
18:00–20:00,—,—,—,—,—
20:00–22:00,—,—,—,—,—
22:00–00:00,—,—,—,—,EMBA (run 1)BAN01
00:00–02:00,—,—,—,—,EMBA (run 2)BAN01
02:00–04:00,—,—,—,PURG (run 1)INJ02,—
05:00–07:00,—,—,REBA (run 1)INJ01,PURG (run 2)INJ02,—
07:00–09:00,—,—,REBA (run 2)INJ02,MOLD (run 1)INJ01,MOLD (run 2)INJ01


,Monday,Tuesday,Wednesday,Thursday,Friday
18:00–20:00,—,—,—,—,—
20:00–22:00,—,—,—,—,—
22:00–00:00,—,—,—,—,—
00:00–02:00,—,—,—,—,—
02:00–04:00,—,—,CTQ (run 2)CTQ01,RETI (run 1)CNC01,FRES (run 2)CNC01
05:00–07:00,—,—,TORN (run 1)CNC01,RETI (run 2)CNC01,ACAB (run 1)CNC01
07:00–09:00,—,CTQ (run 1)CTQ01,TORN (run 2)CNC01,FRES (run 1)CNC01,ACAB (run 2)CNC01


,Monday,Tuesday,Wednesday,Thursday,Friday
18:00–20:00,—,—,—,—,—
20:00–22:00,—,—,—,—,—
22:00–00:00,—,—,—,—,—
00:00–02:00,—,—,—,—,—
02:00–04:00,—,—,SOLD (run 2)BAN01,TEST (run 1)BAN01,CABL (run 1)BAN01
05:00–07:00,—,—,APAR (run 1)BAN01,MNTG (run 1)BAN01,CABL (run 2)BAN01
07:00–09:00,—,SOLD (run 1)BAN01,APAR (run 2)BAN01,MNTG (run 2)BAN01,TEST (run 2)BAN01· rem


,Monday,Tuesday,Wednesday,Thursday,Friday
18:00–20:00,—,—,—,—,—
20:00–22:00,—,—,—,—,—
22:00–00:00,—,—,—,—,—
00:00–02:00,—,—,—,—,—
02:00–04:00,—,—,—,LAVA (run 2)CAB01,VERN (run 2)CAB01· rem
05:00–07:00,—,—,PINT (run 1)CAB01,PINT (run 2)CAB01,ETIQ (run 1)CAB01
07:00–09:00,—,—,LAVA (run 1)CAB01,VERN (run 1)CAB01,ETIQ (run 2)CAB01


In [138]:
real_svg_path = export_schedule_svg(
    real_best_solution,
    "factory_real_schedule.svg",
    "Best schedule - factory_real.txt"
)

print(f"Schedule exported to: {real_svg_path}")

Schedule exported to: schedule_exports/factory_real_schedule.svg


## 14. Critical analysis

### Was a solution found? 

Yes, for both datasets. The solver returned a first valid schedule for `factory_tiny.txt` in section 9, and `run_scheduler` reports the same for `factory_real.txt` above. Both were independently re-checked with `validate_solution`, which reported zero errors.

### Execution time

Finding a *single* valid schedule is fast for both datasets (see the printed `execution_time` / `solve_time` values above). 

Enumerating multiple solutions and scoring them takes longer, since it means walking the search tree instead of stopping at the first leaf.

### Search-space growth

The tiny dataset has 30 task-runs. 
The real dataset has more lines, tasks and machines, so its search space is substantially larger, even though the same hard constraints keep most of it pruned. 

This is visible in the `solutions_found` / `search_exhaustive` values above: if `search_exhaustive` is `False`, the configured limits (`SOLUTION_SEARCH_COUNT_LIMIT` / `solution_count_limit` and `SOLUTION_SEARCH_TIME_LIMIT` / `search_time_limit`) were hit before the whole space could be enumerated, so the reported solution count is a lower bound, not an exhaustive count.

### Soft-constraint violations

The best-scoring solution for each dataset can still carry some penalty (see the penalty breakdown printed after scoring), mainly because only a limited number of hard-constraint-valid candidates is evaluated, not the full space. 

A larger `solution_count_limit` increases the chance of finding a lower-penalty schedule, at the cost of more search and scoring time.

### Limitations

The scoring only considers the candidates enumerated within the search/time cap, so the reported best schedule is the *best found*, not a guaranteed global optimum.

The penalty weights are also a fixed design choice, so a different factory might prioritize these preferences differently.

### Tiny vs real dataset 

The real dataset has more production lines, tasks, operators and machines, which both enlarges the domain per variable and adds more pairwise constraints (line, machine and operator collisions).

This makes the real-dataset run noticeably heavier than the tiny walkthrough, even though the underlying CSP formulation is identical.

### Development history

The first implementation used a more direct, function-based constraint for "at most three tasks per line per night" and required more than 30 minutes without returning a solution. 

The fix was a custom `Constraint` subclass (`MaxTasksPerNightConstraint`) with forward checking, combined with tighter domain filtering, symmetry-breaking between a task's two runs, and skipping machine-conflict constraints between tasks that can never share a machine. 

Together these reduced the tiny-dataset solve time from "does not finish" to a fraction of a second (when using a 20.000 limit).

## 15. Future improvements

### Improved search heuristics

The order in which variables and domain values are explored could be improved.

Some strategies though of include:

- scheduling the most restricted tasks first;
- prioritizing tasks with only one compatible machine;
- prioritizing tasks whose operators have limited availability;
- testing slots that improve task consecutiveness before less desirable slots.

### Improved software architecture

The implementation currently uses several shared mappings and functions inside the notebook.

A future version could encapsulate the scheduling state in a class such as `FactoryScheduler`. This would reduce the use of global variables, improve maintainability and make it easier to run multiple datasets independently.

### Better visualisation

The final timetable could be improved with:

- colour coding for each production line;
- warnings for soft-constraint violations;
- a graphical summary of machine utilisation;
- a graphical summary of operator workload;
- export to CSV or Excel.

### Additional operational constraints

The model could also be extended to support:

- task precedence;
- setup times between tasks;
- different task durations;
- operator breaks;
- machine capacity;
- production priorities;
- minimum intervals between repeated executions;
- emergency maintenance periods.

## 16. Conclusion

This project implemented an intelligent agent for generating the weekly night-shift production schedule of a factory.

The scheduling problem was modelled as a Constraint Satisfaction Problem using Python, Jupyter Notebook and the `python-constraint` library. Each weekly task execution was represented as a variable, while the corresponding domain contained valid combinations of time slot and machine.

The hard constraints ensured that the generated schedules respected production-line capacity, machine compatibility, maintenance periods, operator availability, mandatory machine assignments and resource conflicts.

An evaluation function was used to compare feasible schedules according to soft preferences, including the separation of repeated task-runs, the concentration of each line into four nights, task consecutiveness and reduced machine usage.

The final schedule contains all required task-runs and produced zero validation errors. It is therefore a feasible solution to the scheduling problem.

The development process also demonstrated the computational difficulty of CSP scheduling problems. The initial search required excessive execution time, making it necessary to improve domain filtering, reduce redundant constraints, introduce forward checking and limit candidate enumeration.

The final result is the best solution found within the configured search limits. Although global optimality is not guaranteed, the agent successfully produces a valid and automatically generated production schedule.

The project also demonstrated how Jupyter Notebook can combine documentation, implementation, execution results and critical analysis in a single reproducible file.